# ST-OMR Meter V5-2B — 2-AI / 3-AI Adaptation

This notebook consumes only the completed 300 TRAIN full-meter BBoxes. The first 30 immutable seeds are diagnostic-only and never receive gradient updates. 4-AI is frozen. Threshold tuning, VAL, FINAL_HOLDOUT, Resolver wiring and production promotion remain closed.

In [ ]:
# SETUP / dependency gate
import importlib.metadata, os, subprocess, sys, time
from pathlib import Path

EXPECTED_PILLOW = '12.3.0'
actual_pillow = importlib.metadata.version('Pillow')
print('PILLOW=', actual_pillow)
if actual_pillow != EXPECTED_PILLOW:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-cache-dir', '--no-deps', f'Pillow=={EXPECTED_PILLOW}'])
    raise RuntimeError('Pillow pin installed. Restart the Colab runtime once, then rerun this cell.')

REPO = Path('/content/st-omr-training')
BRANCH = 'agent/meter-v5-2b-23-adaptation-training'
if not REPO.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/khfy7wpr5p-maker/st-omr-training.git', str(REPO)])
else:
    subprocess.check_call(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH, '--depth', '1'])
    subprocess.check_call(['git', '-C', str(REPO), 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', str(REPO), 'reset', '--hard', f'origin/{BRANCH}'])
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
head = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('SETUP=PASS')
print('HEAD=', head)
print('TRAINING=CLOSED_UNTIL_DERIVATION_PASS | VAL=CLOSED | FINAL_HOLDOUT=LOCKED | 4-AI=FROZEN')

In [ ]:
# Mount Drive and bind paths
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = Path('/content/drive/MyDrive/TEST/METER_V2_1500_PACKAGE_AB_CLEAN')
CHECKPOINT_ROOT = Path('/content/drive/MyDrive/ST-OMR-METER-SPECIALISTS')
assert DATA_ROOT.is_dir(), DATA_ROOT
assert CHECKPOINT_ROOT.is_dir(), CHECKPOINT_ROOT
print('DRIVE_BINDING=PASS')

In [ ]:
# Human-QA attestation + deterministic 600-slot derivation with live monitor
import threading, traceback
from IPython.display import clear_output
from st_omr_training import meter_v5_2b_specialist_adaptation as v52b

state = {'phase': 'starting', 'done': False, 'error': None, 'result': None}
def work():
    try:
        state['phase'] = 'bind human QA attestation'
        qa = v52b.write_human_qa_attestation(DATA_ROOT, confirmation=v52b.HUMAN_QA_CONFIRMATION)
        state['phase'] = 'derive 600 approved staff-relative slots'
        manifest = v52b.derive_staff_relative_slots_v1(DATA_ROOT)
        state['result'] = (qa, manifest)
    except BaseException as exc:
        state['error'] = exc
    finally:
        state['done'] = True

thread = threading.Thread(target=work, daemon=False)
thread.start()
started = time.time()
while thread.is_alive():
    clear_output(wait=True)
    print('V5-2B SLOT DERIVATION IZLEME')
    print('Durum: RUNNING')
    print('Faz:', state['phase'])
    print('Gecen sure:', int(time.time()-started), 's')
    print('Guvenlik: TRAINING=CLOSED | VAL=CLOSED | FINAL_HOLDOUT=LOCKED | 4-AI=FROZEN')
    thread.join(timeout=5)
thread.join()
clear_output(wait=True)
if state['error'] is not None:
    print('Durum: FAIL-CLOSED')
    raise state['error']
print('Durum: PASS')
print('QA_ATTESTATION=', state['result'][0])
print('SLOT_MANIFEST=', state['result'][1])
print('TRAINING_BOUNDARY=OPEN_FOR_2_AI_3_AI_ONLY')
print('VAL=CLOSED | FINAL_HOLDOUT=LOCKED | 4-AI=FROZEN')

In [ ]:
# Locate exact frozen historical checkpoints by SHA
DIGIT2 = v52b.locate_checkpoint_by_sha_v1(CHECKPOINT_ROOT, v52b.DIGIT2_SHA256)
DIGIT3 = v52b.locate_checkpoint_by_sha_v1(CHECKPOINT_ROOT, v52b.DIGIT3_SHA256)
DIGIT4 = v52b.locate_checkpoint_by_sha_v1(CHECKPOINT_ROOT, v52b.DIGIT4_SHA256)
print('CHECKPOINT_BINDING=PASS')
print('2-AI=', DIGIT2)
print('3-AI=', DIGIT3)
print('4-AI(FROZEN)=', DIGIT4)

In [ ]:
# Deterministic CPU-only 2-AI / 3-AI adaptation with live monitor
train_state = {'phase': 'fixed training run', 'done': False, 'error': None, 'report': None}
def train_work():
    try:
        train_state['report'] = v52b.train_adapted_specialists_v1(
            DATA_ROOT, digit2_checkpoint=DIGIT2, digit3_checkpoint=DIGIT3
        )
    except BaseException as exc:
        train_state['error'] = exc
    finally:
        train_state['done'] = True

thread = threading.Thread(target=train_work, daemon=False)
thread.start()
started = time.time()
while thread.is_alive():
    clear_output(wait=True)
    print('V5-2B 2/3 ADAPTATION TRAINING IZLEME')
    print('Durum: RUNNING')
    print('Faz:', train_state['phase'])
    print('Gecen sure:', int(time.time()-started), 's')
    print('Config: CPU | epochs=12 | batch=64 | lr=1e-4 | NO_SWEEP | NO_THRESHOLD_TUNING')
    print('Guvenlik: seed30_gradient=0 | VAL=CLOSED | FINAL_HOLDOUT=LOCKED | 4-AI=FROZEN')
    thread.join(timeout=5)
thread.join()
clear_output(wait=True)
if train_state['error'] is not None:
    print('Durum: FAIL-CLOSED')
    raise train_state['error']
report = train_state['report']
print('TRAINING=PASS')
for digit in ('2','3'):
    c = report['candidates'][digit]
    print(digit+'-AI candidate:', c['candidate_path'])
    print('  state_fingerprint=', c['state_fingerprint'])
    print('  train_metrics=', c['final_train_metrics_at_frozen_threshold'])
print('VAL=CLOSED | FINAL_HOLDOUT=LOCKED | 4-AI=FROZEN')

In [ ]:
# 30 untouched diagnostic-seed gate
candidate2 = Path(report['candidates']['2']['candidate_path'])
candidate3 = Path(report['candidates']['3']['candidate_path'])
gate = v52b.evaluate_diagnostic_gate_v1(
    DATA_ROOT,
    digit2_candidate=candidate2,
    digit3_candidate=candidate3,
    digit4_checkpoint=DIGIT4,
)
print('V5-2B DIAGNOSTIC GATE=', gate['gate'])
print('PER_METER_PASS=', gate['per_meter_pass'])
print('DENOMINATOR_EXACT4=', gate['denominator_exact4'], '/30')
print('REASONS=', gate['reasons'])
print('VALIDATION_BBOX_STAGE_AUTHORIZED=', gate['validation_bbox_stage_authorized'])
print('VALIDATION_OPENED=', gate['validation_opened'])
print('FINAL_HOLDOUT_LOCKED=', gate['final_holdout_locked'])
print('PRODUCTION_PROMOTION=', gate['production_promotion_authorized'])